# MNIST Handwritten Digit Classification with a CNN (PyTorch)

**Pipeline:** MNIST → Preprocess → CNN → Train → Evaluate → Confusion Matrix → Confident Misclassifications → Augmentation Comparison

**Goal:** >99% test accuracy on the 10,000 MNIST test images.

## 1. Setup

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Load MNIST and Preprocess

MNIST has 60,000 training and 10,000 test images (28×28 grayscale, 10 classes).

**Preprocessing:** `ToTensor()` converts pixels from `[0, 255]` to `[0.0, 1.0]` and shapes each image as `(1, 28, 28)`.
`Normalize(mean=0.1307, std=0.3081)` then standardizes using the MNIST training-set mean and standard deviation, which keeps the inputs centered and speeds up convergence.

In [ ]:
MEAN, STD = 0.1307, 0.3081
BATCH_SIZE = 128

base_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

train_set = datasets.MNIST("./data", train=True,  download=True, transform=base_tf)
test_set  = datasets.MNIST("./data", train=False, download=True, transform=base_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=1000,       shuffle=False, num_workers=2)

print("Train samples:", len(train_set), "| Test samples:", len(test_set))
print("Image shape:", tuple(train_set[0][0].shape))

### 2.1 Look at the data

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
raw = datasets.MNIST("./data", train=True, download=False)  # un-normalized, for display
for i, ax in enumerate(axes.flat):
    img, label = raw[i]
    ax.imshow(img, cmap="gray"); ax.set_title(label); ax.axis("off")
plt.suptitle("Sample training images"); plt.tight_layout(); plt.show()

# Class balance
counts = np.bincount(raw.targets.numpy(), minlength=10)
plt.figure(figsize=(7, 3))
plt.bar(range(10), counts); plt.xticks(range(10))
plt.xlabel("Digit"); plt.ylabel("Count"); plt.title("Training class distribution")
plt.show()

## 3. CNN Architecture

```
Input (1×28×28)
 → Conv2D(1→32, 3×3, pad=1) + ReLU   → 32×28×28
 → MaxPool 2×2                       → 32×14×14
 → Conv2D(32→64, 3×3, pad=1) + ReLU  → 64×14×14
 → MaxPool 2×2                       → 64×7×7
 → Flatten                           → 3136
 → Dense(3136→128) + ReLU + Dropout(0.5)
 → Dense(128→10)                     → class logits
```

The network outputs raw logits. `nn.CrossEntropyLoss` applies log-softmax internally, so softmax is only applied explicitly when we need probabilities (e.g. to measure prediction confidence).

In [ ]:
class CNN(nn.Module):
    def __init__(self, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.drop  = nn.Dropout(dropout)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)

model = CNN().to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Training

* **Loss:** Cross-Entropy
* **Optimizer:** Adam (lr = 1e-3) with a step learning-rate decay
* **Epochs:** 12 (fixed in advance; the test set is not used to pick a checkpoint)

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss(reduction="sum")
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total_loss += criterion(out, y).item()
            correct += (out.argmax(1) == y).sum().item()
            n += y.size(0)
    return total_loss / n, correct / n


def train_model(loader, epochs=12, lr=1e-3, label="baseline"):
    torch.manual_seed(SEED)
    model = CNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
    hist = {"train_loss": [], "test_loss": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running, seen = 0.0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            running += loss.item() * y.size(0)
            seen += y.size(0)
        scheduler.step()

        test_loss, test_acc = evaluate(model, test_loader)
        hist["train_loss"].append(running / seen)
        hist["test_loss"].append(test_loss)
        hist["test_acc"].append(test_acc)
        print(f"[{label}] epoch {epoch:2d}/{epochs} | "
              f"train loss {running/seen:.4f} | test loss {test_loss:.4f} | test acc {test_acc*100:.2f}%")
    return model, hist


model, hist = train_model(train_loader, label="baseline")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(hist["train_loss"], label="train"); ax[0].plot(hist["test_loss"], label="test")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot([a * 100 for a in hist["test_acc"]], marker="o")
ax[1].axhline(99, color="r", ls="--", label="99% target")
ax[1].set_title("Test accuracy (%)"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Evaluation on the Test Set

In [ ]:
def predict_all(model, loader):
    model.eval()
    probs_list, labels_list = [], []
    with torch.no_grad():
        for x, y in loader:
            probs_list.append(F.softmax(model(x.to(device)), dim=1).cpu())
            labels_list.append(y)
    return torch.cat(probs_list), torch.cat(labels_list)

probs, labels = predict_all(model, test_loader)
preds = probs.argmax(1)
acc = (preds == labels).float().mean().item()
print(f"Test accuracy: {acc*100:.2f}%  ->  {'TARGET MET' if acc > 0.99 else 'below 99% target'}\n")
print(classification_report(labels, preds, digits=4))

### 5.1 Confusion matrix (10×10)

In [ ]:
def plot_cm(labels, preds, title):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8, 6.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=range(10), yticklabels=range(10))
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title)
    plt.show()
    return cm

cm = plot_cm(labels, preds, "Confusion matrix - baseline CNN")

# Most common confusions
off = cm.copy(); np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(-off.ravel())[:5], off.shape))[0]
print("Top confusions (true -> predicted : count):")
for t, p in top:
    print(f"  {t} -> {p} : {off[t, p]}")

## 6. High-Confidence Misclassifications

For every wrongly classified test image, we take the softmax probability of the predicted class as its *confidence* and show the ones where the model was most sure and still wrong. These are typically ambiguous or unusually written digits, or possible label noise.

In [ ]:
confidence, _ = probs.max(1)
wrong_idx = (preds != labels).nonzero(as_tuple=True)[0]
print(f"Total misclassified: {len(wrong_idx)} / {len(labels)}")

order = confidence[wrong_idx].argsort(descending=True)
top_wrong = wrong_idx[order][:10]

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
raw_test = datasets.MNIST("./data", train=False, download=False)
for ax, i in zip(axes.flat, top_wrong):
    ax.imshow(raw_test[i.item()][0], cmap="gray")
    ax.set_title(f"True: {labels[i].item()}  Pred: {preds[i].item()}\nConf: {confidence[i].item()*100:.1f}%", fontsize=10)
    ax.axis("off")
plt.suptitle("Most confident misclassifications"); plt.tight_layout(); plt.show()

## 7. Data Augmentation with Affine Transformations

We train an identical CNN (same seed, architecture, optimizer, epochs) but apply random affine transforms **to the training images only**:

* rotation: ±15°
* translation: up to 10% in x and y
* scaling: 0.9× to 1.1×

Test images are left untouched for the standard comparison. Because MNIST is already very clean, augmentation may give only a small gain on the clean test set, so we also evaluate both models on a **perturbed test set** (random affine transforms applied to test images) to measure robustness.

In [ ]:
affine = transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1))

aug_tf = transforms.Compose([affine, transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,))])
aug_train_set = datasets.MNIST("./data", train=True, download=False, transform=aug_tf)
aug_loader = DataLoader(aug_train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Preview augmented samples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for j in range(8):
    img, label = raw[j]
    axes[0, j].imshow(img, cmap="gray"); axes[0, j].set_title(f"orig {label}"); axes[0, j].axis("off")
    axes[1, j].imshow(affine(img), cmap="gray"); axes[1, j].set_title("augmented"); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
model_aug, hist_aug = train_model(aug_loader, label="augmented")

In [ ]:
# Clean test set results
probs_aug, _ = predict_all(model_aug, test_loader)
preds_aug = probs_aug.argmax(1)
acc_aug = (preds_aug == labels).float().mean().item()

# Perturbed test set (fixed seed so both models see the same transforms)
pert_tf = transforms.Compose([
    transforms.RandomAffine(degrees=20, translate=(0.12, 0.12), scale=(0.85, 1.15)),
    transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,)),
])
pert_set = datasets.MNIST("./data", train=False, download=False, transform=pert_tf)

def eval_perturbed(m):
    torch.manual_seed(123)
    loader = DataLoader(pert_set, batch_size=1000, shuffle=False, num_workers=0)
    return evaluate(m, loader)[1]

pert_base = eval_perturbed(model)
pert_aug  = eval_perturbed(model_aug)

print(f"{'Model':<12}{'Clean test acc':>16}{'Perturbed test acc':>22}")
print(f"{'Baseline':<12}{acc*100:>15.2f}%{pert_base*100:>21.2f}%")
print(f"{'Augmented':<12}{acc_aug*100:>15.2f}%{pert_aug*100:>21.2f}%")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([a*100 for a in hist["test_acc"]], label="baseline", marker="o")
ax[0].plot([a*100 for a in hist_aug["test_acc"]], label="augmented", marker="s")
ax[0].set_title("Clean test accuracy per epoch (%)"); ax[0].set_xlabel("Epoch"); ax[0].legend()

names = ["Clean test", "Perturbed test"]
x = np.arange(2); w = 0.35
ax[1].bar(x - w/2, [acc*100, pert_base*100], w, label="baseline")
ax[1].bar(x + w/2, [acc_aug*100, pert_aug*100], w, label="augmented")
ax[1].set_xticks(x); ax[1].set_xticklabels(names)
lo = min(acc, acc_aug, pert_base, pert_aug) * 100 - 2
ax[1].set_ylim(lo, 100); ax[1].set_ylabel("Accuracy (%)"); ax[1].set_title("Baseline vs augmented"); ax[1].legend()
plt.tight_layout(); plt.show()

_ = plot_cm(labels, preds_aug, "Confusion matrix - augmented CNN")
print(f"Misclassified on clean test: baseline {(preds != labels).sum().item()} | augmented {(preds_aug != labels).sum().item()}")

## 8. Summary

Fill in after running: report baseline vs augmented accuracy (clean and perturbed), which digit pairs were confused most, and what the high-confidence errors look like.

**What the pipeline does:** pixels scaled to `[0,1]` and standardized → two Conv+ReLU+MaxPool blocks extract spatial features → dense layer with dropout classifies → softmax/cross-entropy trains the weights via Adam → evaluation via accuracy, confusion matrix and error analysis → affine augmentation compared against the baseline.